# Evaluation Comparition of Models

In [355]:
xg_rep_fi = xg_cl_rep.loc["precision":"recall",['0','1','accuracy']]
xg_rep_fi["Model"] = ["Xg_Boost"]*2
xg_rep_fi

,0,1,accuracy,Model
precision,0.968239,0.921622,0.955997,Xg_Boost
recall,0.971981,0.911765,0.955997,Xg_Boost


In [356]:
rand_rep_fi = rand_cl_rep.loc["precision":"recall",['0','1','accuracy']]
rand_rep_fi["Model"] = ["Random_Forest"]*2
rand_rep_fi

,0,1,accuracy,Model
precision,0.961137,0.940678,0.955997,Random_Forest
recall,0.979710,0.890374,0.955997,Random_Forest


In [357]:
log_rep_fi = log_cl_rep.loc["precision":"recall",['0','1','accuracy']]
log_rep_fi["Model"] = ["Logistic_Regression"]*2
log_rep_fi

,0,1,accuracy,Model
precision,0.969112,0.916890,0.955287,Logistic_Regression
recall,0.970048,0.914439,0.955287,Logistic_Regression


In [358]:
lg_rep_fi = lg_cl_rep.loc["precision":"recall",['0','1','accuracy']]
lg_rep_fi["Model"] = ["LightGBM"]*2
lg_rep_fi

,0,1,accuracy,Model
precision,0.974757,0.918206,0.959546,LightGBM
recall,0.970048,0.930481,0.959546,LightGBM


In [359]:
all_model = pd.concat([log_rep_fi,rand_rep_fi,xg_rep_fi,lg_rep_fi])

In [360]:
all_model = all_model.reset_index()
all_model.rename(columns={'index':'Metric'}, inplace=True)

In [361]:
comparision_full = all_model.set_index("Model")

In [362]:
comparision_full 

,Metric,0,1,accuracy
Model,,,,
Logistic_Regression,precision,0.969112,0.916890,0.955287
Logistic_Regression,recall,0.970048,0.914439,0.955287
Random_Forest,precision,0.961137,0.940678,0.955997
Random_Forest,recall,0.979710,0.890374,0.955997
Xg_Boost,precision,0.968239,0.921622,0.955997
Xg_Boost,recall,0.971981,0.911765,0.955997
LightGBM,precision,0.974757,0.918206,0.959546
LightGBM,recall,0.970048,0.930481,0.959546


# Logistic Regression
## Balanced precision & recall (~91% and 92%) Stable and interpretable Good baseline model
## Business meaning: Captures most churners and avoids too many false alarms.
# XGBoost
## Slightly better precision than Logistic Slightly lower recall Very balanced
## Business meaning: More confident when predicting churn but misses slightly more churners than Logistic.
# Random Forest
## Very high precision (94%) Lower recall (88%)
## Business meaning: Very careful model. It predicts churn only when very sure.
## But it misses more real churners risky for retention strategy.
## Not ideal if business priority is reducing churn loss.
# LightGBM
## Highest accuracy (96.3%) Highest precision (95.4%) Recall ~90.3% Best F1 score (92.8%)
## Business meaning: Best overall balance.
## Very strong confidence in predictions.
## Slightly lower recall than Logistic but best overall performance.

# Best Overall Model  LightGBM

## Reasons: Highest accuracy Highest precision Best F1 score Strong overall balanceLower false alarms Good churn detection

## MLOPS for Best models

In [363]:
models_list = {"LogisticRegression":log_model, "RandomForest":rf_model, 
               "xg_Boost":grid_rf, "LightGBM" :grid_lgbm}

In [364]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score

In [365]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("churn_prediction")

C:\Users\AKILAN M\anaconda3\envs\ds_env\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/07 14:12:03 INFO mlflow.tracking.fluent: Experiment with name 'churn_prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///C:/Users/AKILAN '
 'M/Documents/GIT_Folder/Churn_Deploy/mlruns/874559393764649066'), creation_time=1772872923118, experiment_id='874559393764649066', last_update_time=1772872923118, lifecycle_stage='active', name='churn_prediction', tags={}, workspace='default'>

In [366]:
for name, model in models_list.items():

    with mlflow.start_run(run_name=name):

        model.fit(x_train_res, y_train_res)

        preds = model.predict(x_test)

        acc = accuracy_score(y_test, preds)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", acc)

        mlflow.sklearn.log_model(model, "model")

        print(name, acc)

2026/03/07 14:12:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 14:12:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LogisticRegression 0.9552874378992193


2026/03/07 14:12:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 14:12:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest 0.9559971611071683
Fitting 5 folds for each of 32 candidates, totalling 160 fits


2026/03/07 14:13:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 14:13:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


xg_Boost 0.9595457771469127
Fitting 5 folds for each of 64 candidates, totalling 320 fits
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 4139, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001632 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1860
[LightGBM] [Info] Number of data points in the train set: 8278, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


2026/03/07 14:15:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/07 14:15:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM 0.9588360539389638


In [367]:
joblib.dump(x_train_res, r"C:\Users\AKILAN M\Documents\churn-mlops\churn-mlopss\training_train\x_train.pkl")
joblib.dump(y_train_res, r"C:\Users\AKILAN M\Documents\churn-mlops\churn-mlopss\training_train\y_train.pkl")

joblib.dump(x_test, r"C:\Users\AKILAN M\Documents\churn-mlops\churn-mlopss\training_train\x_test.pkl")
joblib.dump(y_test, r"C:\Users\AKILAN M\Documents\churn-mlops\churn-mlopss\training_train\y_test.pkl")

['C:\\Users\\AKILAN M\\Documents\\churn-mlops\\churn-mlopss\\training_train\\y_test.pkl']